In [1]:
# stats for each (flux dates, phenocam dates, frequency, etc)
# located ecoregion level 3

In [17]:
import os

script_dir = os.path.dirname(os.path.abspath('load_utils.py'))

In [18]:
script_dir

'/Users/alexfache/Documents/GitHub/PLSP/code/sites_info'

In [19]:
from pathlib import Path
DIR_PLANET = Path('~').expanduser() / 'Library/CloudStorage/Dropbox' / 'planet'

In [20]:
import geopandas as gpd
import pandas as pd

# Load Site Data CSVs


In [21]:
phenocam_df = pd.read_csv(DIR_PLANET / 'qgis/sites/phenocam_sites.csv')
phenocam_df.rename(columns={c: f'(phenocam){c}' for c in list(phenocam_df.columns) if c not in ['site_name']}, inplace=True)
# phenocam_df

In [22]:
flux_df = pd.read_csv(DIR_PLANET / 'qgis/sites/AmeriFlux-site-search-results-202605032116.csv')
flux_df.rename(columns={c: f'(flux){c}' for c in list(flux_df.columns) if c not in ['Site ID']}, inplace=True)
# flux_df

In [23]:
study_sites_df = pd.read_csv(DIR_PLANET / 'qgis/sites/study_sites.csv')
study_sites_df[['latitude', 'longitude']] = study_sites_df['site_marker'].str.split(',', expand=True)
study_sites_df['latitude'] = study_sites_df['latitude'].astype(float)
study_sites_df['longitude'] = study_sites_df['longitude'].astype(float)
# study_sites_df

In [24]:
data_gdf = gpd.GeoDataFrame(study_sites_df, geometry=gpd.points_from_xy(study_sites_df.latitude, study_sites_df.longitude))
data_gdf = pd.merge(data_gdf, flux_df, on=['Site ID'], how='left')
data_gdf = pd.merge(data_gdf, phenocam_df, on=['site_name'], how='left')
data_gdf.rename(columns={'site_name': '(phenocam)site_name'}, inplace=True)
data_gdf.drop(columns=['(flux)Latitude (degrees)', '(flux)Longitude (degrees)'], axis=1, inplace=True)

# data_gdf

In [25]:
import json


def split_lat_lon_string(js):
    data = json.loads(js)
    return list(zip(data['lat'], data['lng']))


data_gdf['site_polygon'] = data_gdf['site_polygon'].apply(split_lat_lon_string)

In [26]:
data_gdf.to_file(DIR_PLANET / 'sites.gpkg', driver='GPKG')

/opt/miniconda3/envs/LCSC/lib/python3.12/site-packages/pyogrio/geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [28]:
data_gdf.iloc[0]['site_polygon']

[(31.7814910017996, -109.994712510874),
 (31.7814910017996, -109.889087489126),
 (31.6915089982004, -109.889087489126),
 (31.6915089982004, -109.994712510874),
 (31.7814910017996, -109.994712510874)]

In [14]:
data_gdf.columns

Index(['Flux Site Name', 'Site ID', '(phenocam)site_name', 'site_url',
       'site_marker', 'site_polygon', 'latitude', 'longitude', 'geometry',
       '(flux)Name', '(flux)Principal Investigator', '(flux)Data Use Policy',
       '(flux)AmeriFlux BASE Data', '(flux)AmeriFlux FLUXNET Data',
       '(flux)Vegetation Abbreviation (IGBP)',
       '(flux)Vegetation Description (IGBP)',
       '(flux)Climate Class Abbreviation (Koeppen)',
       '(flux)Climate Class Description (Koeppen)',
       '(flux)Mean Average Precipitation (mm)',
       '(flux)Mean Average Temperature (degrees C)', '(flux)Country',
       '(flux)Elevation (m)', '(flux)Number of years of AmeriFlux BASE data',
       '(flux)AmeriFlux BASE Data Start', '(flux)AmeriFlux BASE Data End',
       '(flux)Years of AmeriFlux BASE Data', '(flux)AmeriFlux BASE DOI',
       '(flux)AmeriFlux FLUXNET Data Start',
       '(flux)AmeriFlux FLUXNET Data End',
       '(flux)Years of AmeriFlux FLUXNET Data', '(flux)AmeriFlux FLUXNET DOI',

# Visualization of Sites


In [16]:
from ipyleaflet import AwesomeIcon, Map, Marker, Polygon, ScaleControl, basemaps
from ipywidgets import HTML, Layout

site_icon = AwesomeIcon(name='tower-cell', marker_color='blue', icon_color='blue', spin=False)
# phenocam_icon = AwesomeIcon(name='camera', marker_color='green', icon_color='green', spin=False)

center = (44, -103)
m = Map(center=center, zoom=5, basemap=basemaps.Esri.WorldImagery, layout=Layout(height='500px'))
m.add(ScaleControl(position='bottomleft'))


site_boundary_locations = []

for index, site in data_gdf.iterrows():
    # Build hover tooltip HTML
    flux_name = site.get('(flux)Name', 'N/A')
    site_id = site.get('Site ID', 'N/A')
    veg = site.get('(flux)Vegetation Abbreviation (IGBP)', 'N/A')
    base_start = site.get('(flux)AmeriFlux BASE Data Start', 'N/A')
    base_end = site.get('(flux)AmeriFlux BASE Data End', 'N/A')
    flux_start = site.get('(flux)AmeriFlux FLUXNET Data Start', 'N/A')
    flux_end = site.get('(flux)AmeriFlux FLUXNET Data End', 'N/A')
    pheno_name = site.get('(phenocam)site_name', 'N/A')
    pheno_start = site.get('(phenocam)date_first', 'N/A')
    pheno_end = site.get('(phenocam)date_last', 'N/A')

    tooltip_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; min-width: 180px;">
        <b style="font-size: 13px;">Flux: {flux_name} ({site_id})</b><hr style="margin: 4px 0;">
        <b>Vegetation:</b> {veg}<br>
        <b>Base data:</b> {base_start} - {base_end}<br>
        <b>Flux data:</b> {flux_start} - {flux_end}<br>
        <hr style="margin: 4px 0;">
        <b>PhenoCam:</b> {pheno_name}<br>
        <b>PhenoCam data:</b> {pheno_start} - {pheno_end}
    </div>
    """

    marker = Marker(
        icon=site_icon,
        location=(site['latitude'], site['longitude']),
        draggable=True,
        title=f"{flux_name} ({site_id})",
    )

    # Attach popup that opens on hover
    popup = HTML(value=tooltip_html)
    marker.popup = popup

    m.add(marker)
    site_boundary_locations.append(site['site_polygon'])

    # m.add(Marker(icon=phenocam_icon, location=(site['(phenocam)latitude'], site['(phenocam)longitude']), draggable=True, title=f'{site.get("(phenocam)site_name", "N/A")}'))

# Add polygons
site_boundary_polygons = Polygon(locations=site_boundary_locations, color='blue', fill_color='blue', fill_opacity=0.1, weight=2)
m.add(site_boundary_polygons)

display(m)

m.save('sites.html', title='My Map')

Map(center=[44, -103], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

# Create Selected Sites and Info CSV


In [ ]:
def underscore_name(row):
    return '_'.join(row.split(' '))

In [23]:
selected_sites = data_gdf[['(flux)Name']]
selected_sites.rename(columns={'(flux)Name': 'name'}, inplace=True)

selected_sites['site'] = selected_sites['name'].apply(underscore_name)

/var/folders/v_/1p9kfw391j985gnp_mvlhp880000gn/T/ipykernel_32162/3007803465.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sites.rename(columns={'(flux)Name': 'name'}, inplace=True)
/var/folders/v_/1p9kfw391j985gnp_mvlhp880000gn/T/ipykernel_32162/3007803465.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sites['site'] = selected_sites['name'].apply(underscore_name)


In [24]:
selected_sites.head(10)

,name,site
0,Walnut Gulch Kendall Grasslands,Walnut_Gulch_Kendall_Grasslands
1,Willard Juniper Savannah,Willard_Juniper_Savannah
2,Walnut Gulch Lucky Hills Shrub,Walnut_Gulch_Lucky_Hills_Shrub
3,Santa Rita Mesquite,Santa_Rita_Mesquite
4,Santa Rita Grassland,Santa_Rita_Grassland
5,Sevilleta shrubland,Sevilleta_shrubland
6,Sevilleta grassland,Sevilleta_grassland
7,Mountainair Pinyon-Juniper Woodland,Mountainair_Pinyon-Juniper_Woodland
8,Konza Prairie LTER (KNZ),Konza_Prairie_LTER_(KNZ)
9,ARM Southern Great Plains site- Lamont,ARM_Southern_Great_Plains_site-_Lamont
